# Radar (Spider) Charts — QTL Results per Noise Type & Strength

Reproduces the radar chart style for every combination of
`noise_type` × `noise_strength` in `qtl.csv`, overlaying **No ZNE** (blue) and
**With ZNE** (orange) metric profiles.

**Metrics plotted:** accuracy · precision · recall · specificity · sensitivity · f1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import warnings, os
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})

os.makedirs('radar_plots', exist_ok=True)
print("Output folder: radar_plots/")

## 1. Load Data

In [ ]:
df = pd.read_csv('qtl.csv')
print("Columns:", df.columns.tolist())
df.head(10)

## 2. Metric Configuration

In [ ]:
# Six metrics to plot and their display labels
METRICS = ['accuracy', 'precision', 'recall', 'specificity', 'sensitivity', 'f1']
METRIC_LABELS = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Sensitivity', 'F1']

# Map each metric key to its CSV column name
NO_ZNE_COLS = {
    'accuracy':    'accuracy_no_zne',
    'precision':   'precision_no_zne',
    'recall':      'recall_no_zne',
    'specificity': 'specificity_no_zne',
    'sensitivity': 'sensitivity_no_zne',
    'f1':          'f1_no_zne',
}
ZNE_COLS = {
    'accuracy':    'accuracy_zne',
    'precision':   'precision_zne',
    'recall':      'recall_zne',
    'specificity': 'specificity_zne',
    'sensitivity': 'sensitivity_zne',
    'f1':          'f1_zne',
}

COLOR_NO_ZNE = '#1f77b4'   # blue
COLOR_ZNE    = '#ff7f0e'   # orange
FILL_ALPHA   = 0.18
GRID_LEVELS  = [0.2, 0.4, 0.6, 0.8, 1.0]

print("Metric config ready.")

## 3. Radar Chart Helper

In [ ]:
def make_radar(ax, values_no_zne, values_zne, labels, title):
    """Draw a single radar chart on *ax* for 6 metrics."""
    N = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()

    # Close the polygon
    v1 = values_no_zne + [values_no_zne[0]]
    v2 = values_zne    + [values_zne[0]]
    a  = angles        + [angles[0]]

    # grid rings
    for level in GRID_LEVELS:
        ring = [level] * (N + 1)
        ax.plot(a, ring, color='#bbbbbb', linewidth=0.6, linestyle='-', zorder=1)
        # level label on the last spoke (F1)
        ax.text(angles[-1], level + 0.04, f"{level:.1f}",
                ha='center', va='bottom', fontsize=7.5, color='#666666')

    # spokes
    for angle in angles:
        ax.plot([angle, angle], [0, 1.0],
                color='#bbbbbb', linewidth=0.6, zorder=1)

    # filled polygons
    ax.fill(a, v1, color=COLOR_NO_ZNE, alpha=FILL_ALPHA)
    ax.fill(a, v2, color=COLOR_ZNE,    alpha=FILL_ALPHA)

    # outlines
    ax.plot(a, v1, color=COLOR_NO_ZNE, linewidth=1.8, zorder=3)
    ax.plot(a, v2, color=COLOR_ZNE,    linewidth=1.8, zorder=4)

    # outer boundary circle
    theta_full = np.linspace(0, 2 * np.pi, 300)
    ax.plot(theta_full, [1.0] * 300, color='black', linewidth=1.0, zorder=2)

    # axis labels
    for angle, label in zip(angles, labels):
        ax.text(angle, 1.20, label,
                ha='center', va='center',
                fontsize=10, fontweight='normal')

    ax.set_ylim(0, 1.38)
    ax.set_title(title, pad=22, fontsize=11, fontweight='bold')
    ax.axis('off')


print("Helper defined.")

## 4. Individual Radar Charts — one per (noise_type x noise_strength)

In [ ]:
legend_elements = [
    Line2D([0], [0], color=COLOR_NO_ZNE, linewidth=2, label='No ZNE'),
    Line2D([0], [0], color=COLOR_ZNE,    linewidth=2, label='With ZNE'),
]

for _, row in df.iterrows():
    nt = row['noise_type']
    p  = row['noise_strength']

    vals_no_zne = [row[NO_ZNE_COLS[m]] for m in METRICS]
    vals_zne    = [row[ZNE_COLS[m]]    for m in METRICS]

    title = f"{nt.capitalize()} Noise  |  p = {p}"

    fig = plt.figure(figsize=(6, 6))
    ax  = fig.add_subplot(111, polar=True)

    make_radar(ax, vals_no_zne, vals_zne, METRIC_LABELS, title)

    fig.legend(
        handles=legend_elements,
        loc='upper right',
        bbox_to_anchor=(1.28, 0.98),
        frameon=True,
        fontsize=10
    )

    fname = f"radar_plots/{nt}_p{str(p).replace('.', '')}.png"
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved -> {fname}")

## 5. Combined Grid — All Conditions in One Figure

In [ ]:
noise_types = df['noise_type'].unique()
strengths   = sorted(df['noise_strength'].unique())

nrows = len(noise_types)
ncols = len(strengths)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(6 * ncols, 5.5 * nrows),
    subplot_kw=dict(polar=True)
)

# Ensure axes is always 2-D
if nrows == 1: axes = axes[np.newaxis, :]
if ncols == 1: axes = axes[:, np.newaxis]

for r, nt in enumerate(noise_types):
    for c, p in enumerate(strengths):
        row_data = df[(df['noise_type'] == nt) & (df['noise_strength'] == p)]
        if row_data.empty:
            axes[r, c].axis('off')
            continue

        row = row_data.iloc[0]
        vals_no_zne = [row[NO_ZNE_COLS[m]] for m in METRICS]
        vals_zne    = [row[ZNE_COLS[m]]    for m in METRICS]
        title       = f"{nt.capitalize()} Noise  |  p = {p}"

        make_radar(axes[r, c], vals_no_zne, vals_zne, METRIC_LABELS, title)

# Shared legend
fig.legend(
    handles=legend_elements,
    loc='upper right',
    bbox_to_anchor=(1.01, 0.99),
    fontsize=12,
    frameon=True
)

fig.suptitle(
    'QTL Metric Radar Charts — No ZNE vs With ZNE\nAll Noise Types & Strengths',
    fontsize=15, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig('radar_plots/radar_all_combined.png', dpi=300, bbox_inches='tight')
print("Saved -> radar_plots/radar_all_combined.png")
plt.show()

## 6. Summary of Saved Files

In [ ]:
import glob
files = sorted(glob.glob('radar_plots/*.png'))
print(f"Total files saved: {len(files)}")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f}  ({size_kb:.1f} KB)")